# LLM Serving with Ray Serve and vLLM

This notebook demonstrates how to deploy large language models for inference using Ray Serve with vLLM integration. You'll learn how to:

- Deploy LLMs with vLLM for high-throughput inference
- Create OpenAI-compatible API endpoints
- Configure autoscaling for production workloads
- Serve multiple models on a single cluster
- Use tensor parallelism for large models

## Prerequisites

- 1+ GPUs with at least 16GB VRAM (for 7B models)
- Python 3.9+
- HuggingFace account for gated models

## 1. Installation

In [ ]:
!pip install -q "ray[serve]" vllm transformers

## 2. Setup and Configuration

In [ ]:
import os
import sys

sys.path.insert(0, ".")

from utils import (
    print_gpu_status,
    detect_gpus,
    init_ray,
    ClusterMode,
)

In [ ]:
# Check GPU availability
print_gpu_status()

gpu_info = detect_gpus()
NUM_GPUS = gpu_info["count"] if gpu_info["available"] else 0
print(f"\nAvailable GPUs: {NUM_GPUS}")

In [ ]:
# Configuration
CLUSTER_MODE = ClusterMode.LOCAL  # Change to ClusterMode.ANYSCALE for Anyscale

# Model configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small model for demo
# MODEL_NAME = "microsoft/phi-2"  # Alternative: 2.7B
# MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"  # Larger model
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"  # Mistral

# Serving configuration
TENSOR_PARALLEL_SIZE = 1  # Number of GPUs for tensor parallelism
MAX_MODEL_LEN = 2048  # Maximum sequence length
GPU_MEMORY_UTILIZATION = 0.9  # Fraction of GPU memory to use

# Autoscaling configuration
MIN_REPLICAS = 1
MAX_REPLICAS = 4
TARGET_NUM_ONGOING_REQUESTS = 10  # Requests per replica before scaling

## 3. Initialize Ray

In [ ]:
init_ray(mode=CLUSTER_MODE)

## 4. Create vLLM Deployment

Ray Serve deployments wrap your model and handle:
- Request batching
- Autoscaling
- Health checks
- Load balancing

In [ ]:
import ray
from ray import serve
from typing import Dict, List, Optional, AsyncGenerator
import asyncio


@serve.deployment(
    ray_actor_options={"num_gpus": TENSOR_PARALLEL_SIZE},
    autoscaling_config={
        "min_replicas": MIN_REPLICAS,
        "max_replicas": MAX_REPLICAS,
        "target_num_ongoing_requests_per_replica": TARGET_NUM_ONGOING_REQUESTS,
    },
)
class VLLMDeployment:
    def __init__(self, model_name: str, **vllm_kwargs):
        from vllm import LLM, SamplingParams
        
        self.model_name = model_name
        self.llm = LLM(
            model=model_name,
            **vllm_kwargs,
        )
        self.default_sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.95,
            max_tokens=256,
        )
    
    async def generate(self, request: Dict) -> Dict:
        """
        Generate completion for a single prompt.
        
        Args:
            request: Dict with 'prompt' and optional sampling parameters
        
        Returns:
            Dict with 'text' containing generated response
        """
        from vllm import SamplingParams
        
        prompt = request.get("prompt", "")
        
        # Extract sampling parameters from request
        sampling_params = SamplingParams(
            temperature=request.get("temperature", 0.7),
            top_p=request.get("top_p", 0.95),
            max_tokens=request.get("max_tokens", 256),
            stop=request.get("stop", None),
        )
        
        outputs = self.llm.generate([prompt], sampling_params)
        generated_text = outputs[0].outputs[0].text
        
        return {
            "text": generated_text,
            "model": self.model_name,
            "usage": {
                "prompt_tokens": len(outputs[0].prompt_token_ids),
                "completion_tokens": len(outputs[0].outputs[0].token_ids),
            }
        }
    
    async def batch_generate(self, requests: List[Dict]) -> List[Dict]:
        """
        Generate completions for multiple prompts efficiently.
        
        vLLM batches these internally for optimal throughput.
        """
        from vllm import SamplingParams
        
        prompts = [r.get("prompt", "") for r in requests]
        
        # Use default params for batch (or extend to support per-request params)
        outputs = self.llm.generate(prompts, self.default_sampling_params)
        
        results = []
        for output in outputs:
            results.append({
                "text": output.outputs[0].text,
                "model": self.model_name,
            })
        
        return results

## 5. Create OpenAI-Compatible API

This wrapper provides an API compatible with the OpenAI client library.

In [ ]:
from starlette.requests import Request
from starlette.responses import JSONResponse, StreamingResponse
import json
import time


@serve.deployment
@serve.ingress(serve.APIIngress)
class OpenAICompatibleAPI:
    """
    OpenAI-compatible API endpoint.
    
    Supports:
    - POST /v1/completions
    - POST /v1/chat/completions
    """
    
    def __init__(self, vllm_deployment):
        self.vllm = vllm_deployment
    
    @serve.api(route="/v1/completions", methods=["POST"])
    async def completions(self, request: Request) -> JSONResponse:
        """OpenAI Completions API compatible endpoint."""
        body = await request.json()
        
        prompt = body.get("prompt", "")
        
        result = await self.vllm.generate.remote({
            "prompt": prompt,
            "temperature": body.get("temperature", 0.7),
            "max_tokens": body.get("max_tokens", 256),
            "top_p": body.get("top_p", 0.95),
            "stop": body.get("stop"),
        })
        
        # Format as OpenAI response
        response = {
            "id": f"cmpl-{int(time.time())}",
            "object": "text_completion",
            "created": int(time.time()),
            "model": result["model"],
            "choices": [{
                "text": result["text"],
                "index": 0,
                "finish_reason": "stop",
            }],
            "usage": result.get("usage", {}),
        }
        
        return JSONResponse(response)
    
    @serve.api(route="/v1/chat/completions", methods=["POST"])
    async def chat_completions(self, request: Request) -> JSONResponse:
        """OpenAI Chat Completions API compatible endpoint."""
        body = await request.json()
        
        messages = body.get("messages", [])
        
        # Convert chat messages to prompt
        prompt = self._format_chat_prompt(messages)
        
        result = await self.vllm.generate.remote({
            "prompt": prompt,
            "temperature": body.get("temperature", 0.7),
            "max_tokens": body.get("max_tokens", 256),
            "top_p": body.get("top_p", 0.95),
            "stop": body.get("stop"),
        })
        
        # Format as OpenAI chat response
        response = {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": result["model"],
            "choices": [{
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": result["text"],
                },
                "finish_reason": "stop",
            }],
            "usage": result.get("usage", {}),
        }
        
        return JSONResponse(response)
    
    def _format_chat_prompt(self, messages: List[Dict]) -> str:
        """Convert chat messages to a prompt string."""
        # Simple formatting - adjust based on your model's expected format
        prompt_parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            
            if role == "system":
                prompt_parts.append(f"System: {content}")
            elif role == "user":
                prompt_parts.append(f"User: {content}")
            elif role == "assistant":
                prompt_parts.append(f"Assistant: {content}")
        
        prompt_parts.append("Assistant:")
        return "\n\n".join(prompt_parts)

## 6. Deploy the Model

In [ ]:
# Create deployment with vLLM configuration
vllm_deployment = VLLMDeployment.bind(
    model_name=MODEL_NAME,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    trust_remote_code=True,
)

# Create OpenAI-compatible API
api = OpenAICompatibleAPI.bind(vllm_deployment)

print(f"Deployment configured for model: {MODEL_NAME}")

In [ ]:
# Deploy to Ray Serve
handle = serve.run(api, name="llm-api")

print("\nDeployment started!")
print(f"API endpoint: http://localhost:8000")
print(f"Dashboard: http://localhost:8265")

## 7. Test the API

### Direct Python Client

In [ ]:
import requests

# Test completions endpoint
response = requests.post(
    "http://localhost:8000/v1/completions",
    json={
        "prompt": "Write a haiku about machine learning:",
        "max_tokens": 50,
        "temperature": 0.8,
    }
)

print("Completions API Response:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Test chat completions endpoint
response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is Ray Serve used for?"}
        ],
        "max_tokens": 100,
        "temperature": 0.7,
    }
)

print("Chat Completions API Response:")
print(json.dumps(response.json(), indent=2))

### Using OpenAI Python Client

In [ ]:
# Install OpenAI client if needed
!pip install -q openai

In [ ]:
from openai import OpenAI

# Point to local Ray Serve endpoint
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",  # API key not required for local deployment
)

# Use like normal OpenAI client
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": "Write a Python function to reverse a string."}
    ],
    max_tokens=150,
)

print("OpenAI Client Response:")
print(response.choices[0].message.content)

## 8. Monitor and Scale

Ray Serve provides built-in monitoring through the dashboard.

In [ ]:
# Check deployment status
print(serve.status())

In [ ]:
# Send concurrent requests to trigger autoscaling
import asyncio
import aiohttp

async def send_request(session, prompt):
    async with session.post(
        "http://localhost:8000/v1/completions",
        json={"prompt": prompt, "max_tokens": 50}
    ) as response:
        return await response.json()

async def load_test(num_requests=20):
    async with aiohttp.ClientSession() as session:
        prompts = [f"Tell me about topic {i}:" for i in range(num_requests)]
        tasks = [send_request(session, p) for p in prompts]
        results = await asyncio.gather(*tasks)
        return results

# Run load test
print("Running load test with 20 concurrent requests...")
results = await load_test(20)
print(f"Completed {len(results)} requests")

## 9. Advanced: Multi-Model Serving

Serve multiple models on the same cluster with a router.

In [ ]:
# Example: Multi-model router (conceptual)
"""
@serve.deployment
class ModelRouter:
    def __init__(self, models: Dict[str, serve.DeploymentHandle]):
        self.models = models
    
    async def route(self, request: Dict) -> Dict:
        model_name = request.get("model", "default")
        if model_name not in self.models:
            model_name = "default"
        
        return await self.models[model_name].generate.remote(request)

# Create multiple model deployments
small_model = VLLMDeployment.bind(model_name="TinyLlama/TinyLlama-1.1B")
large_model = VLLMDeployment.bind(model_name="meta-llama/Llama-2-7b")

# Create router
router = ModelRouter.bind({
    "tinyllama": small_model,
    "llama2": large_model,
    "default": small_model,
})
"""
print("See code cell for multi-model serving example")

## 10. Cleanup

In [ ]:
# Shutdown the deployment
serve.shutdown()
print("Ray Serve shutdown complete")

In [ ]:
from utils import shutdown_ray
shutdown_ray()

## Next Steps

- **Production deployment**: Use `serve.run` with a YAML config for production
- **Streaming**: Implement streaming responses for better UX
- **Authentication**: Add API key authentication for production
- **Caching**: Add response caching for repeated queries
- **Batch inference**: See `batch_inference_llm.ipynb` for offline batch processing

## Resources

- [Ray Serve LLM Documentation](https://docs.ray.io/en/latest/serve/llm/index.html)
- [vLLM Documentation](https://docs.vllm.ai/)
- [Ray Serve Autoscaling](https://docs.ray.io/en/latest/serve/scaling-and-resource-allocation.html)
- [Anyscale RayLLM](https://www.anyscale.com/product/library/ray-llm)